## pcFVA Plots for G6PD

Necessary imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rbc_gem_utils import read_cobra_model
from rbc_gem_utils.analysis.overlay import load_overlay_model

### Centralize file paths

In [2]:
model_id = "RBC3P_expanded"
dataset_name = "RBComics_G6PD"
data_path = Path("../../../../data/analysis/OVERLAY").resolve()
root_path = Path("../../../..").resolve()
results_path = root_path / "data" / "processed" / model_id / "OVERLAY"
pcmodel_dirpath = data_path / model_id
dataset_path = results_path / dataset_name
dataset_models_dirpath = dataset_path / "pcmodels"
pcfva_results_dirpath = dataset_path / "pcFVA"
corr_results_dirpath = dataset_path / "correlations"
df_pcfva_all_filename = pcfva_results_dirpath / f"{model_id}_PC_FVAresults_ALL.tsv"
model_filename = pcmodel_dirpath / f"{model_id}.xml"
pcmodel_filename = pcmodel_dirpath / f"{model_id}_PC.xml"

print(results_path)
print(pcmodel_dirpath)
print(dataset_path)
print(dataset_models_dirpath)
print(pcfva_results_dirpath)
print(corr_results_dirpath)
print(df_pcfva_all_filename)
print(model_filename)
print(pcmodel_filename)

D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcmodels
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcFVA
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\correlations
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcFVA\RBC3P_expanded_PC_FVAresults_ALL.tsv
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded\RBC3P_expanded.xml
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded\RBC3P_expanded_PC.xml


### Load main models

In [3]:
model = read_cobra_model(filename=model_filename)
pcmodel = load_overlay_model(filename=pcmodel_filename)

Set parameter Username


### Load all results from pcFVA

In [4]:
df_pcfva_all = pd.read_csv(df_pcfva_all_filename, sep="\t")
df_pcfva_all

,reactions,optimum,model,minimum,maximum
0,ADA,0.00,RBC3P_expanded_PC_Mean_D10,0.0,0.278274
1,ADA,0.00,RBC3P_expanded_PC_Mean_D23,0.0,0.282364
2,ADA,0.00,RBC3P_expanded_PC_Mean_D42,0.0,0.277745
3,ADA,0.00,RBC3P_expanded_PC_Median_D10,0.0,0.273331
4,ADA,0.00,RBC3P_expanded_PC_Median_D23,0.0,0.274004
...,...,...,...,...,...
7922875,TPI,0.99,RBC3P_expanded_PC_S00649_D23,0.0,0.001753
7922876,TPI,0.99,RBC3P_expanded_PC_S00649_D42,0.0,0.001790
7922877,TPI,0.99,RBC3P_expanded_PC_S00650_D10,0.0,0.001541
7922878,TPI,0.99,RBC3P_expanded_PC_S00650_D23,0.0,0.001325


### Separate reaction dataframe by reaction type

In [5]:
# Initialize entries with prefixes used for seperating DataFrames
dict_of_dataframes_types = {
    "reactions": None,
    "proteins": "PROTDL",
    # "complexes": "CPLXFM",
    # "complex_dilutions": "CPLXDL",
    "enzymes": "ENZDL",
    # "enzyme_formation": "ENZFM",
    "budgets": "PBDL",
    "relaxation": "RELAX",
}
for key, prefix in dict_of_dataframes_types.copy().items():
    if prefix:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x.startswith(prefix))
        ]
    else:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x in model.reactions)
        ]
    dict_of_dataframes_types[key] = df.copy()

## Correlations

### Always abundance independent reactions

In [8]:
groupby_list = ["model", "reactions"]
always_abundance_independent = [
    r.id for r in model.reactions.query(lambda x: not x.boundary and not x.genes)
]
print(
    f"Number of reactions w/o genes, always expression independent: {len(always_abundance_independent)}"
)
always_abundance_independent

Number of reactions w/o genes, always expression independent: 7


['H2O2t', 'Ht', 'PIt', 'NTPA', 'FE2O2OX', 'FE2H2O2X', 'HOXG']

### Get maximum flux range

#### Get maximum flux in either direction

In [11]:
df = dict_of_dataframes_types["reactions"].copy()
df = df.groupby(groupby_list)[["minimum", "maximum"]].agg(
    {
        "minimum": "min",
        "maximum": "max",
    }
)
df_max_flux_per_model = df.abs().max(axis=1)
df_max_flux_per_model.name = "Flux"
df_max_flux_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA          0.278274
                              ADEt         0.018362
                              ADK1         0.018362
                              ADNK1        0.018362
                              ADNt         0.278274
                                             ...   
RBC3P_expanded_PC_S00650_D42  SPODM        0.171777
                              TALA         0.041669
                              TKT1         0.041669
                              TKT2         0.041669
                              TPI          0.045783
Name: Flux, Length: 177898, dtype: float64

#### Get range of maximum flux

In [12]:
df = dict_of_dataframes_types["reactions"].copy()
df["Range"] = df["maximum"] - df["minimum"]
df_flux_range_per_model = df.groupby(groupby_list)["Range"].max()
df_flux_range_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA          0.278274
                              ADEt         0.018362
                              ADK1         0.018362
                              ADNK1        0.018362
                              ADNt         0.296636
                                             ...   
RBC3P_expanded_PC_S00650_D42  SPODM        0.171777
                              TALA         0.082190
                              TKT1         0.082190
                              TKT2         0.082190
                              TPI          0.045783
Name: Range, Length: 177898, dtype: float64

### Get maximum abundance

In [13]:
min_reaction_list = model.reactions.query(lambda x: x.gene_reaction_rule).list_attr(
    "id"
)
reaction_enzymes_map = {
    rid: tuple(
        pcmodel.reactions.query(
            lambda x: x.id.startswith(f"ENZDL_enzyme_{rid}_")
        ).list_attr("id")
    )
    for rid in min_reaction_list
}
enzyme_reaction_map = {
    enzyme: rid for rid, enzymes in reaction_enzymes_map.items() for enzyme in enzymes
}
df = dict_of_dataframes_types["enzymes"].copy()
df["reactions"] = df["reactions"].apply(lambda x: enzyme_reaction_map[x])
df_max_enzyme_per_model = df.groupby(groupby_list)["maximum"].max()
df_max_enzyme_per_model.name = "Abundance"
df_max_enzyme_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA           2.089841
                              ADEt          4.387130
                              ADK1         22.533588
                              ADNK1         2.487410
                              ADNt          3.092031
                                             ...    
RBC3P_expanded_PC_S00650_D42  SPODM        28.826502
                              TALA          2.055036
                              TKT1          0.847117
                              TKT2          0.847117
                              TPI          12.508601
Name: Abundance, Length: 111874, dtype: float64

### Merge results above into one DataFrame

In [14]:
df_reaction_flux_expression = (
    pd.merge(
        df_max_flux_per_model,
        df_flux_range_per_model,
        left_index=True,
        right_index=True,
    )
    .merge(df_max_enzyme_per_model, left_index=True, right_index=True)
    .reset_index(drop=False)
)
df_reaction_flux_expression

,model,reactions,Flux,Range,Abundance
0,RBC3P_expanded_PC_Mean_D10,ADA,0.278274,0.278274,2.089841
1,RBC3P_expanded_PC_Mean_D10,ADEt,0.018362,0.018362,4.387130
2,RBC3P_expanded_PC_Mean_D10,ADK1,0.018362,0.018362,22.533588
3,RBC3P_expanded_PC_Mean_D10,ADNK1,0.018362,0.018362,2.487410
4,RBC3P_expanded_PC_Mean_D10,ADNt,0.278274,0.296636,3.092031
...,...,...,...,...,...
111869,RBC3P_expanded_PC_S00650_D42,SPODM,0.171777,0.171777,28.826502
111870,RBC3P_expanded_PC_S00650_D42,TALA,0.041669,0.082190,2.055036
111871,RBC3P_expanded_PC_S00650_D42,TKT1,0.041669,0.082190,0.847117
111872,RBC3P_expanded_PC_S00650_D42,TKT2,0.041669,0.082190,0.847117
